In [61]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")


In [62]:
### Load the Ann Train_model,scaler Pickle file , onehotencoder

model =load_model('model.h5')

In [63]:
## load the scaler and encoder

with open('onehot_encoder_geo.pkl','rb') as file:
    onehot_encoder_geo =pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [64]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [65]:
# One-hot encode  'Geography' 
geo_encoder = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoder_df =pd.DataFrame(geo_encoder,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoder_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [66]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [67]:
## Encode categorical Variable
input_df ['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [68]:
## concation one hot encoding
input_df = pd.concat([input_df.drop("Geography",axis=1),geo_encoder_df],axis =1)

In [69]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [70]:
## scaling the data
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [71]:
## predict the churn 
prediction = model.predict(input_scaled)
prediction

1/1 [==============================] - 0s 245ms/step


array([[0.0326311]], dtype=float32)

In [72]:
prediction_proba =prediction[0][0]

In [73]:
prediction_proba

0.032631103

In [74]:
if prediction_proba >0.5:
    print("The customer likely to churn"),

else:
    print("The customer is not likely to churn")

The customer is not likely to churn
